In [1]:
pip install openai chromadb sentence-transformers

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
import json
import chromadb
from chromadb.utils import embedding_functions
from openai import OpenAI

# ==========================================
# 1. 초기 설정 (API 키 및 경로)
# ==========================================
os.environ["OPENAI_API_KEY"] = "API 키 입력" # 본인 API 키 입력
client = OpenAI()

# 기존 데이터 경로 설정
DATA_PATH = "../data/cards"

# 한국어 임베딩 모델 설정
ko_embedding_func = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="jhgan/ko-sroberta-multitask"
)

# Vector DB 연결
chroma_client = chromadb.PersistentClient(path="./card_vector_db")

# 기존 컬렉션 삭제 (임베딩 충돌 방지 및 최신화용)
try:
    chroma_client.delete_collection(name="card_info_v2")
except:
    pass

collection = chroma_client.create_collection(
    name="card_info_v2", 
    embedding_function=ko_embedding_func
)

# ==========================================
# 2. 기존 JSON 데이터 로드 및 인덱싱
# ==========================================
def indexing_existing_data():
    file_list = [f for f in os.listdir(DATA_PATH) if f.endswith('.json')]
    
    ids, documents, metadatas = [], [], []

    print(f"[{DATA_PATH}] 경로에서 데이터를 읽어오는 중...")

    for i, file_name in enumerate(file_list):
        with open(os.path.join(DATA_PATH, file_name), 'r', encoding='utf-8') as f:
            card = json.load(f)
            
            # RAG 성능을 높이기 위해 모든 정보를 문장으로 결합 (Context 생성)
            context_text = (
                f"카드사: {card['company']}, 카드명: {card['card_name']}. "
                f"캐시백: {card.get('cashback', '정보없음')}, "
                f"주요혜택처: {card.get('benefit_place', '정보없음')}, "
                f"추가할인: {card.get('discount', '정보없음')}, "
                f"전월실적조건: {card.get('performance', '전월실적 없음')}, "
                f"해외사용: {card.get('overseas', '정보없음')}."
            )
            
            ids.append(f"card_{i}")
            documents.append(context_text)
            metadatas.append({"name": card['card_name'], "company": card['company']})

    collection.add(ids=ids, documents=documents, metadatas=metadatas)
    print(f"✅ {len(documents)}개의 카드 정보가 Vector DB에 등록되었습니다.")

# ==========================================
# 3. RAG 답변 생성 함수
# ==========================================
def get_ai_response(query, persona="", use_rag=True):
    """
    use_rag=True 이면 RAG 적용 답변, False 이면 순수 GPT 답변 (평가 비교용)
    """
    if use_rag:
        # DB에서 관련 정보 검색 (Top 3)
        results = collection.query(query_texts=[query], n_results=3)
        retrieved_context = "\n".join(results['documents'][0])
        
        system_content = f"""
        당신은 카드 추천 전문 비서입니다. 사용자의 페르소나를 고려하여 답변하세요.
        페르소나: {persona}
        
        반드시 제공된 [카드 데이터]의 내용에만 기반하여 답변하세요. 
        데이터에 없는 내용은 지어내지 말고 모른다고 답하세요.

        [카드 데이터]
        {retrieved_context}
        """
    else:
        # Base GPT-3.5 성능 확인용
        system_content = f"당신은 카드 추천 비서입니다. 페르소나({persona})에 맞춰 답변하세요."

    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[
            {"role": "system", "content": system_content},
            {"role": "user", "content": query}
        ],
        temperature=0.3
    )
    return response.choices[0].message.content

# ==========================================
# 4. 평가를 위한 실행부
# ==========================================
indexing_existing_data()

# 테스트 페르소나 예시
persona_A = "사회초년생, 배달앱과 스타벅스를 자주 이용함, 실적 압박이 없는 카드를 원함"
user_query = "나에게 맞는 체크카드를 추천해줘."

print("\n" + "="*50)
print(f"실험 질문: {user_query}")
print(f"페르소나: {persona_A}")
print("="*50)

# 1) Base 모델 답변 (RAG 미적용)
print("\n[1. Base GPT-3.5 답변]")
print(get_ai_response(user_query, persona_A, use_rag=False))

# 2) 고도화 모델 답변 (RAG 적용)
print("\n[2. RAG 적용 GPT-3.5 답변]")
print(get_ai_response(user_query, persona_A, use_rag=True))

[../data/cards] 경로에서 데이터를 읽어오는 중...
✅ 409개의 카드 정보가 Vector DB에 등록되었습니다.

실험 질문: 나에게 맞는 체크카드를 추천해줘.
페르소나: 사회초년생, 배달앱과 스타벅스를 자주 이용함, 실적 압박이 없는 카드를 원함

[1. Base GPT-3.5 답변]
안녕하세요! 페르소나에 맞는 체크카드를 추천해드릴게요. 

1. 삼성카드 The 삼성체크카드
- 스타벅스 이용 시 혜택이 있습니다. 매월 1회 무료 음료 혜택이 제공됩니다.
- 배달앱 이용 시에도 적립이 가능하며, 다양한 가맹점에서 혜택을 받을 수 있습니다.
- 실적 압박이 없어서 부담 없이 사용할 수 있습니다.

2. 신한카드 The Live 체크카드
- 스타벅스 이용 시 할인 혜택이 있습니다.
- 배달앱 결제 시 캐시백 혜택을 받을 수 있습니다.
- 연회비가 없어 부담 없이 사용할 수 있습니다.

이 두 가지 카드 중에서 선택하시면 편리하게 일상 속에서 혜택을 누리실 수 있을 것 같아요. 어떤 카드를 선택하시겠어요?

[2. RAG 적용 GPT-3.5 답변]
사회초년생이시고 배달앱과 스타벅스를 자주 이용하시는 페르소나에게는 IBK기업은행의 '일상의 기쁨 Dream 체크카드'가 적합할 것 같습니다. 이 카드는 편의점에서 5% 할인과 커피 전문점에서 10% 할인 혜택을 제공하며, 버스와 지하철 이용 시 건당 100원 할인 혜택도 받을 수 있습니다. 또한, 전월실적이 30만원 이상이어야 하는 압박이 없고, 해외사용이 필요하지 않으시다면 이 카드가 적합할 것입니다.
